# Terra Signal: comparação de técnicas de modelagem (Grupo 1)

Estudo de qual técnica usar para prever o churn com as dummies definidas nos notebooks 06 a 08.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- **Técnicas:** regressão logística, árvore de decisão e boosting.
- **Variáveis:** as 25 dummies do notebook 07, sem segmentação (notebook 08).
- **Fora do modelo:** `CustomerFeedback`, `MonthlyIncome`, `MonthlyCharges`, `TotalCharges` e `PhoneService`.
- **Fora deste estudo:** o ponto de corte e o arquivo `prediction.csv`.
- Roda com pandas, numpy, scipy, statsmodels e scikit-learn, sem Spark. Funciona localmente e no Databricks (Git folder).

## Setup

In [1]:
%pip install -q statsmodels  # o ambiente serverless do Databricks pode não ter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [2]:
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


In [3]:
def prep(d):
    """Regras do notebook 06: recupera tenure. TotalCharges só serve para isso e não entra no modelo."""
    d = d.copy()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")

Clientes no treino: 5633 | churners: 1503 (26.7%)


In [4]:
def melhor_corte(x, y, min_folha):
    """Corte binário de maior qui-quadrado, com p-valor corrigido por Bonferroni pelo nº de cortes testados."""
    tab = pd.DataFrame({"x": x, "y": y}).groupby("x")["y"].agg(["size", "sum"]).sort_index()
    n_esq, c_esq = tab["size"].cumsum().values, tab["sum"].cumsum().values
    n, c = n_esq[-1], c_esq[-1]
    validos = (n_esq >= min_folha) & (n - n_esq >= min_folha)
    if not validos.any():
        return None
    a, b = c_esq, n_esq - c_esq
    e, f = c - c_esq, (n - n_esq) - (c - c_esq)
    den = (a + b) * (e + f) * (a + e) * (b + f)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(den > 0, n * (a * f - b * e) ** 2 / den, 0.0)
    q = np.where(validos, q, -1.0)
    i = int(np.argmax(q))
    return float(tab.index[i]), float(q[i]), min(1.0, chi2.sf(q[i], 1) * int(validos.sum()))


def aid(x, y, min_folha=0.05, alfa=0.05, max_faixas=None):
    """AID do notebook 07: divide sempre a faixa com maior qui-quadrado."""
    x, y = np.asarray(x, float), np.asarray(y)
    tam = int(np.ceil(min_folha * len(x)))
    nos, cortes = [np.ones(len(x), bool)], []
    while not max_faixas or len(nos) < max_faixas:
        candidatos = []
        for k, m in enumerate(nos):
            r = melhor_corte(x[m], y[m], tam)
            if r and r[2] < alfa:
                candidatos.append((r[1], k, r))
        if not candidatos:
            break
        _, k, r = max(candidatos, key=lambda z: z[0])
        m = nos.pop(k)
        nos += [m & (x <= r[0]), m & (x > r[0])]
        cortes.append(r[0])
    return sorted(cortes)


DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_faixas(x, cortes, ref):
    """Uma dummy por faixa de tenure, exceto a faixa de referência `ref`."""
    f = np.digitize(x, cortes, right=True)
    limites = [-np.inf] + list(cortes) + [np.inf]
    return pd.DataFrame(
        {f"tenure=({limites[i]:g}, {limites[i + 1]:g}]": (f == i).astype(int) for i in range(len(limites) - 1) if i != ref},
        index=x.index,
    )


def montar_dummies(d, cortes, ref):
    cat = pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )
    return pd.concat([cat, dummies_faixas(d["tenure_num"], cortes, ref)], axis=1)


def faixa_referencia(x, cortes):
    return int(pd.Series(np.digitize(x, cortes, right=True)).value_counts().idxmax())

### 1. Quais colunas originais alimentam o modelo?

In [5]:
ORIGEM = list(DUMMIES_CATEG) + ["tenure"]
FORA = [c for c in hist_raw.columns if c not in ORIGEM and c != "Churn"]
cortes_base = aid(hist["tenure_num"], y, max_faixas=6)
X_base = montar_dummies(hist, cortes_base, faixa_referencia(hist["tenure_num"], cortes_base))
print("Entram no modelo:", ORIGEM)
print("Não entram no modelo:", FORA)
print("Dummies:", X_base.shape[1], "| cortes de tenure:", cortes_base)
print("Alguma dummy vem do texto?", any("Feedback" in c for c in X_base.columns))

Entram no modelo: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure']
Não entram no modelo: ['customerID', 'PhoneService', 'MonthlyCharges', 'TotalCharges', 'CustomerFeedback', 'MonthlyIncome']
Dummies: 25 | cortes de tenure: [5.0, 17.0, 43.0, 59.0, 70.0]
Alguma dummy vem do texto? False


**Resposta:** entram 16 colunas originais, que viram 25 dummies. **`CustomerFeedback` não entra**, por decisão do grupo, e nenhuma dummy vem do texto. Também ficam de fora `customerID`, `PhoneService`, `MonthlyCharges`, `MonthlyIncome` e `TotalCharges`. `TotalCharges` e `MonthlyCharges` só são usadas para recuperar o `tenure` dos casos `unknown`.

## Como as técnicas são comparadas

**Validação cruzada aninhada**, com 5 folds e 4 repetições (os mesmos folds para todas as técnicas):

- **Dentro de cada fold de treino:** calculamos os cortes de `tenure` pelo AID e escolhemos os hiperparâmetros com uma validação interna de 3 folds.
- **No fold de teste:** medimos o desempenho, com dados que não participaram de nenhuma dessas escolhas.

**Critério de escolha, definido antes de ver o resultado desta comparação:** a AUC é a métrica principal, porque mede se o modelo coloca os clientes de maior risco no topo da lista. Entre as técnicas cuja diferença para a melhor fica dentro de um desvio-padrão pareado, escolhemos a mais simples, nesta ordem: regressão logística, árvore e boosting. Os testes anteriores (notebooks 07 e 08) já usaram regressão logística, então é uma técnica que conhecíamos melhor ao definir o critério.

As demais métricas ajudam a interpretar:

- **Log-loss e Brier:** medem a qualidade das probabilidades. Quanto menor, melhor.
- **Acurácia e F1 com corte em 0,5:** servem só de referência, porque a TS ainda não informou a métrica de avaliação nem os custos da ação. O corte fica para outra etapa.

### 2. Qual é o desempenho de cada técnica fora da amostra?

In [6]:
TECNICAS = {
    "Logística": (LogisticRegression(max_iter=3000), {"C": [0.01, 0.1, 1, 10]}),
    "Árvore": (DecisionTreeClassifier(random_state=0), {"max_depth": [2, 4, 6, 8, 10, 12], "min_samples_leaf": [20, 50, 100, 250]}),
    "Boosting": (HistGradientBoostingClassifier(learning_rate=0.05, random_state=0), {"max_depth": [1, 2, 3], "max_iter": [100, 300, 600, 1000]}),
}
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
INTERNO = StratifiedKFold(3, shuffle=True, random_state=0)

linhas, escolhas = [], []
previsoes = {k: np.full((4, len(hist)), np.nan) for k in TECNICAS}
for i, (tr, te) in enumerate(CV.split(hist, y)):
    cortes = aid(hist["tenure_num"].iloc[tr], y.iloc[tr], max_faixas=6)
    ref = faixa_referencia(hist["tenure_num"].iloc[tr], cortes)
    Xtr, Xte = montar_dummies(hist.iloc[tr], cortes, ref), montar_dummies(hist.iloc[te], cortes, ref)
    ytr, yte = y.iloc[tr], y.iloc[te]
    modelos = {"Sempre a taxa média": DummyClassifier(strategy="prior").fit(Xtr, ytr)}
    for nome, (est, grade) in TECNICAS.items():
        g = GridSearchCV(est, grade, scoring="roc_auc", cv=INTERNO).fit(Xtr, ytr)
        modelos[nome] = g.best_estimator_
        escolhas.append({"tecnica": nome, **{k: str(v) for k, v in g.best_params_.items()}})
    for nome, m in modelos.items():
        p = m.predict_proba(Xte)[:, 1]
        if nome in previsoes:
            previsoes[nome][i // 5, te] = p
        linhas.append({"tecnica": nome, "fold": i, "AUC": roc_auc_score(yte, p), "log_loss": log_loss(yte, p), "Brier": brier_score_loss(yte, p),
                       "acuracia_0.5": accuracy_score(yte, p >= .5), "F1_0.5": f1_score(yte, p >= .5, zero_division=0)})
res = pd.DataFrame(linhas)
print(res.groupby("tecnica").mean(numeric_only=True).drop(columns="fold").round(4).sort_values("AUC", ascending=False).to_string())
print("\nDesvio-padrão da AUC entre folds:")
print(res.groupby("tecnica")["AUC"].std().round(4).to_string())

                        AUC  log_loss   Brier  acuracia_0.5  F1_0.5
tecnica                                                            
Boosting             0.8449    0.4179  0.1358        0.8006  0.5771
Logística            0.8448    0.4178  0.1356        0.8012  0.5812
Árvore               0.8254    0.5319  0.1428        0.7865  0.5458
Sempre a taxa média  0.5000    0.5801  0.1956        0.7332  0.0000

Desvio-padrão da AUC entre folds:
tecnica
Boosting               0.0085
Logística              0.0085
Sempre a taxa média    0.0000
Árvore                 0.0123


**Resposta:** regressão logística e boosting empatam, e a árvore fica atrás.

| Técnica | AUC | Log-loss | Acurácia (corte 0,5) | F1 (corte 0,5) |
|---|---|---|---|---|
| Boosting | 0,8449 | 0,4179 | 80,1% | 0,577 |
| Regressão logística | 0,8448 | 0,4178 | 80,1% | 0,581 |
| Árvore | 0,8254 | 0,5319 | 78,7% | 0,546 |
| Sempre a taxa média | 0,5000 | 0,5801 | 73,3% | 0,000 |

As três técnicas superam a referência, que prevê sempre a taxa média. A acurácia dessa referência já é de 73,3%, porque ela acerta todos os não churners ao prever sempre `No`.

### 3. Qual técnica o critério escolhe?

In [7]:
auc = res.pivot(index="fold", columns="tecnica", values="AUC")
ORDEM = ["Logística", "Árvore", "Boosting"]
melhor = auc[ORDEM].mean().idxmax()
print("Melhor AUC média:", melhor)
ESCOLHIDA = None
for t in ORDEM:
    dif = auc[melhor] - auc[t]
    dentro = dif.mean() <= dif.std(ddof=0)
    print(f"{t:10s} AUC {auc[t].mean():.4f} | distância da melhor {dif.mean():.4f} ± {dif.std(ddof=0):.4f} | dentro de 1 dp: {dentro}")
    if dentro and ESCOLHIDA is None:
        ESCOLHIDA = t
print("\nEscolhida pelo critério:", ESCOLHIDA)
print("\nHiperparâmetros mais escolhidos nos 20 folds:")
for t in ORDEM:
    e = pd.DataFrame([x for x in escolhas if x["tecnica"] == t]).drop(columns="tecnica").dropna(axis=1)
    print(f"  {t}: {e.astype(str).agg(', '.join, axis=1).value_counts().head(3).to_dict()}")

Melhor AUC média: Boosting
Logística  AUC 0.8448 | distância da melhor 0.0001 ± 0.0010 | dentro de 1 dp: True
Árvore     AUC 0.8254 | distância da melhor 0.0195 ± 0.0075 | dentro de 1 dp: False
Boosting   AUC 0.8449 | distância da melhor 0.0000 ± 0.0000 | dentro de 1 dp: True

Escolhida pelo critério: Logística

Hiperparâmetros mais escolhidos nos 20 folds:
  Logística: {'1': 17, '10': 3}
  Árvore: {'12, 50': 7, '10, 50': 5, '6, 100': 2}
  Boosting: {'1, 1000': 20}


**Resposta:** **regressão logística.** A diferença para o boosting é de 0,0001 ± 0,0010, dentro do ruído, e a logística é a mais simples. A árvore fica fora, com distância de 0,0195 ± 0,0075.

O empate tem uma explicação: o boosting escolheu **profundidade 1 nos 20 folds**. Com essa profundidade, cada árvore usa uma única variável, então o modelo também soma efeitos individuais, como a regressão logística. Isso reforça o piloto de interações, que indicou efeitos basicamente aditivos.

### 4. As probabilidades são confiáveis?

In [8]:
def erro_calibracao(p, y):
    d = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)
    t = pd.DataFrame({"d": d, "p": p, "y": y.values}).groupby("d").mean()
    return float((t["p"] - t["y"]).abs().mean()), t


for t in ORDEM:
    e = np.mean([erro_calibracao(previsoes[t][r], y)[0] for r in range(4)])
    print(f"{t:10s} erro médio de calibração por decil: {e:.4f} | média prevista: {np.nanmean(previsoes[t]):.3f} | valores distintos na 1ª repetição: {len(np.unique(previsoes[t][0].round(6)))}")

P = previsoes[ESCOLHIDA].mean(axis=0)  # média das 4 repetições, sempre com o cliente fora do treino
_, tab = erro_calibracao(P, y)
print(f"\n{ESCOLHIDA}, por decil de risco:")
print(tab.rename(columns={"p": "prob_media_prevista", "y": "churn_observado"}).round(3).to_string())

Logística  erro médio de calibração por decil: 0.0080 | média prevista: 0.267 | valores distintos na 1ª repetição: 5124
Árvore     erro médio de calibração por decil: 0.0212 | média prevista: 0.267 | valores distintos na 1ª repetição: 228
Boosting   erro médio de calibração por decil: 0.0104 | média prevista: 0.267 | valores distintos na 1ª repetição: 5036

Logística, por decil de risco:
   prob_media_prevista  churn_observado
d                                      
0                0.010            0.012
1                0.025            0.025
2                0.048            0.046
3                0.088            0.114
4                0.148            0.138
5                0.231            0.222
6                0.329            0.329
7                0.450            0.433
8                0.582            0.590
9                0.758            0.759


**Resposta:** sim, e a regressão logística é a mais bem calibrada. O erro médio por decil é de 0,8 ponto percentual, contra 1,0 no boosting e 2,1 na árvore. A probabilidade prevista acompanha o churn observado em toda a escala, de 1,0% contra 1,2% no decil mais baixo até 75,8% contra 75,9% no mais alto. A maior diferença está no quarto decil (8,8% contra 11,4%).

Isso importa para a regra financeira, que multiplica a probabilidade pelo valor do cliente. Como não usamos balanceamento de classes, a probabilidade prevista pode ser lida diretamente como risco.

### 5. Quantos churners a lista de maior risco alcança?

In [9]:
ordem = np.argsort(-P)
linhas = []
for frac in [0.10, 0.20, 0.30, 0.40, 0.50]:
    n = int(round(frac * len(P)))
    topo = y.values[ordem[:n]]
    linhas.append({"maior_risco": f"{frac:.0%}", "clientes": n, "churners": int(topo.sum()), "precisao": round(topo.mean(), 3),
                   "captura": round(topo.sum() / y.sum(), 3), "lift": round(topo.mean() / y.mean(), 2)})
print(pd.DataFrame(linhas).to_string(index=False))

maior_risco  clientes  churners  precisao  captura  lift
        10%       563       427     0.758    0.284  2.84
        20%      1127       760     0.674    0.506  2.53
        30%      1690      1004     0.594    0.668  2.23
        40%      2253      1189     0.528    0.791  1.98
        50%      2816      1314     0.467    0.874  1.75

**Resposta:** os 20% de maior risco concentram cerca de metade dos churners.

| Clientes de maior risco | Precisão | Churners alcançados | Lift |
|---|---|---|---|
| 10% | 75,8% | 28,4% | 2,84 |
| 20% | 67,4% | 50,6% | 2,53 |
| 30% | 59,4% | 66,8% | 2,23 |
| 40% | 52,8% | 79,1% | 1,98 |
| 50% | 46,7% | 87,4% | 1,75 |

Para comparar, uma lista aleatória teria precisão de 26,7% em qualquer tamanho.

### 6. O modelo funciona dentro de cada segmento?

In [10]:
SEGMENTOS = {
    "Contrato mensal": hist["Contract"] == "Month-to-month",
    "Contrato anual ou dois anos": hist["Contract"] != "Month-to-month",
    "Fibra": hist["InternetService"] == "Fiber optic",
    "Sem internet": hist["InternetService"] == "No",
    "Mensal e fibra": (hist["Contract"] == "Month-to-month") & (hist["InternetService"] == "Fiber optic"),
    "Tenure até 5 meses": hist["tenure_num"] <= 5,
    "Tenure acima de 43 meses": hist["tenure_num"] > 43,
}
print(pd.DataFrame([{
    "segmento": k, "clientes": int(m.sum()), "churn": round(y[m].mean(), 3), "prob_media": round(P[m].mean(), 3), "AUC": round(roc_auc_score(y[m], P[m]), 3),
} for k, m in SEGMENTOS.items()]).to_string(index=False))

                   segmento  clientes  churn  prob_media   AUC
            Contrato mensal      3099  0.427       0.426 0.761
Contrato anual ou dois anos      2534  0.071       0.072 0.776
                      Fibra      2476  0.420       0.419 0.791
               Sem internet      1233  0.072       0.074 0.834
             Mensal e fibra      1698  0.547       0.545 0.716
         Tenure até 5 meses      1092  0.541       0.537 0.765
   Tenure acima de 43 meses      2033  0.102       0.106 0.795


**Resposta:** sim, mas separa menos onde o risco é mais alto. A probabilidade média acompanha o churn real em todos os segmentos, com diferença de no máximo 0,4 ponto percentual. Já a AUC varia de 0,834 entre clientes sem internet até **0,716 em contrato mensal com fibra**, grupo de 1.698 clientes com 54,7% de churn. Nesse grupo, o modelo tem menos informação para decidir quem priorizar, pois quase todos já compartilham os principais fatores de risco.

### 7. Quais são os pesos do modelo de regressão logística?

In [11]:
aj = sm.Logit(y, sm.add_constant(X_base.astype(float))).fit(disp=0)
ic = np.exp(aj.conf_int())
pesos = pd.DataFrame({"OR": np.exp(aj.params), "IC95_inf": ic[0], "IC95_sup": ic[1], "p": aj.pvalues}).drop("const").round(3).sort_values("OR", ascending=False)
print(pesos.to_string())

                                            OR  IC95_inf  IC95_sup      p
tenure=(-inf, 5]                         4.756     3.792     5.965  0.000
tenure=(5, 17]                           1.744     1.414     2.150  0.000
MultipleLines=No phone service           1.594     1.203     2.113  0.001
StreamingTV=Yes                          1.466     1.229     1.747  0.000
MultipleLines=Yes                        1.386     1.166     1.647  0.000
StreamingMovies=Yes                      1.301     1.092     1.551  0.003
SeniorCitizen=1                          1.292     1.074     1.554  0.007
Partner=Yes                              1.040     0.878     1.233  0.649
DeviceProtection=Yes                     1.032     0.868     1.227  0.723
gender=Female                            1.016     0.881     1.172  0.827
OnlineBackup=Yes                         0.889     0.753     1.050  0.167
Dependents=Yes                           0.857     0.703     1.043  0.124
PaymentMethod=Bank transfer (automatic

**Resposta:** os maiores efeitos vêm de tempo de casa, internet e contrato. O OR compara cada categoria com sua referência, mantendo as demais variáveis constantes.

| Fator | Comparação | OR |
|---|---|---|
| `tenure` até 5 meses | contra 18 a 43 meses | 4,76 |
| `tenure` acima de 70 meses | contra 18 a 43 meses | 0,22 |
| Sem internet | contra fibra | 0,17 |
| DSL | contra fibra | 0,40 |
| Contrato de dois anos | contra mensal | 0,27 |
| Contrato de um ano | contra mensal | 0,50 |
| Com `TechSupport` | contra sem | 0,70 |
| Com `OnlineSecurity` | contra sem | 0,72 |
| Cartão automático | contra cheque eletrônico | 0,71 |

**Isso ajusta uma leitura anterior.** O notebook 03 mostrou que mais serviços adicionais acompanham menos churn. Controlando as demais variáveis, só `TechSupport` e `OnlineSecurity` mantêm essa relação. `OnlineBackup` e `DeviceProtection` não se distinguem de um efeito nulo, e os streamings ficam com OR acima de 1 (1,47 em TV e 1,30 em filmes).

`Partner`, `DeviceProtection`, `gender`, `OnlineBackup` e `Dependents` têm p > 0,10. Esses pesos foram estimados sem regularização, para obter os intervalos de confiança, então diferem um pouco dos pesos do modelo final, que usa C = 1.

### 8. O modelo escolhido funciona no `inference`?

In [12]:
est, grade = TECNICAS[ESCOLHIDA]
ref_base = faixa_referencia(hist["tenure_num"], cortes_base)
final = GridSearchCV(est, grade, scoring="roc_auc", cv=INTERNO).fit(X_base, y)
X_inf = montar_dummies(inf, cortes_base, ref_base)
p_inf = final.predict_proba(X_inf)[:, 1]
print("Hiperparâmetros finais:", final.best_params_)
print("Mesmas colunas do treino:", list(X_inf.columns) == list(X_base.columns), "| clientes pontuados:", len(p_inf), "| probabilidades vazias:", int(np.isnan(p_inf).sum()))
print(f"Probabilidade média: {p_inf.mean():.3f} | churn no history: {y.mean():.3f}")
print("Distribuição:", pd.Series(p_inf).describe().round(3).to_dict())

Hiperparâmetros finais: {'C': 1}
Mesmas colunas do treino: True | clientes pontuados: 1409 | probabilidades vazias: 0
Probabilidade média: 0.265 | churn no history: 0.267
Distribuição: {'count': 1409.0, 'mean': 0.265, 'std': 0.246, 'min': 0.003, '25%': 0.042, '50%': 0.195, '75%': 0.452, 'max': 0.907}


**Resposta:** sim, do ponto de vista técnico. O modelo pontua os 1.409 clientes, com as mesmas colunas do treino e sem valores vazios. A probabilidade média é de 26,5%, próxima dos 26,7% de churn do `history`, o que é coerente com a ausência de drift do notebook 03. Não dá para medir o desempenho nessa base, porque o `inference` não tem `Churn`.

O arquivo `prediction.csv` ainda não foi gerado, porque depende da escolha do ponto de corte.

## Limites desta análise

- **Não há uma base de teste intocada:** os cortes do AID e as decisões dos notebooks 07 e 08 usaram o mesmo `history`. A validação aninhada reduz esse problema, mas o desempenho real pode ser um pouco menor.
- **O limite da grade:** o boosting escolheu 1.000 iterações, o maior valor testado, e a árvore escolheu com frequência a profundidade 12, também a maior. Mais iterações com profundidade 1 tendem a se aproximar de um modelo aditivo como a regressão logística, e a distância da árvore é grande, então não esperamos mudança na conclusão, mas não testamos valores maiores.
- **Sem horizonte de tempo:** a probabilidade indica o quanto o cliente se parece com quem aparece como churner no histórico. Ela não diz "vai sair nos próximos 3 meses", pois a base não tem datas.
- **Definição de `tenure` e de churn:** falta a TS confirmar se o `tenure` para de contar no cancelamento e se os clientes do `inference` estão ativos. Sem nenhuma variável de tempo, a AUC cai cerca de 0,020 (notebook 07).
- **Os pesos não medem o efeito de uma ação:** um OR baixo em `TechSupport` não prova que oferecer suporte reduziria o churn de um cliente. Para isso, seria preciso ter dados de ações de retenção.
- **Métrica da submissão:** a TS ainda não informou como o `prediction.csv` será avaliado, então a acurácia e o F1 com corte em 0,5 são apenas referências.